# Weight of Evidence & Information Value — Home Credit Default Risk

A full practice workbook covering the WOE / IV skills you need to turn raw features into scorecard-ready inputs for a Probability of Default (PD) model, built on `application_train.csv`.

**How to use this notebook**

1. Work through sections in order — WOE from first principles first, then IV, then binning, then transformation, then the pitfalls, then a capstone.
2. Each section gives you brief context and the key syntax, then exercises. Write your code in the empty cells.
3. Every section also has **interpretation questions** — answer them in the markdown cells provided. In scorecard work the *reasoning* (why this binning, why this convention, why this feature survives) is the deliverable — the code is just how you get there.
4. There are **conceptual challenges** scattered throughout marked 🧠. Some have no code at all — they are there to catch the misconceptions that quietly wreck a scorecard. Answer them cold, then check yourself.
5. Bring your attempts to Claude for marking, or ask for hints when stuck. Try each exercise cold first.

**Ground rules**

- **Fix your convention once, in Section 1, and never flip it.** WOE has two sign conventions and mixing them is the single most common WOE bug. Write your choice at the top and obey it everywhere.
- **Learn every bin, every WOE, every IV on the training split only.** The validation rows never touch a WOE calculation until you *apply* a train-learned mapping to them. Leakage here inflates every downstream metric.
- **Every WOE table gets an IV, and every binning gets a monotonicity check.** Build the habit now.
- **State the minimum bin size you will accept before you look at the rates.** Deciding "5% or 200 rows, whichever is larger" *after* seeing a tempting cell rate is how you fool yourself.
- Before computing a WOE table, predict the *shape* — which bins should be safe, which risky. If the WOE trend surprises you, that is a finding. Write it down.

## 0 — Setup, data loading, and the good/bad frame

Run these cells to get started. The load cell follows your usual repo-relative pattern — adjust the path if your data lives elsewhere. This section also sets up the one framing decision everything else depends on: **who is the "bad"**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 130)
pd.set_option("display.width", 160)
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

In [ ]:
from pathlib import Path

def find_repo_root(marker=".git"):
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    return p

# REPO_ROOT = find_repo_root()
# DATA_PATH = REPO_ROOT / "Data" / "home-credit-default-risk" / "application_train.csv"

df = pd.read_csv('../../Datasets/Home Credit Default Risk/application_train.csv')
print(df.shape)

### Derived columns

Reuse the clean columns you built in 1.5 so the WOE work reads cleanly. `DAYS_*` columns are negative day counts; flip them once here.

In [ ]:
# Derived columns (same conventions as your 1.5 workbook)
df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365.25

df['YEARS_EMPLOYED'] = -df['DAYS_EMPLOYED'] / 365.25
# The +365243 sentinel encodes "not employed" (pensioners etc.). Mask it to NaN.
df['YEARS_EMPLOYED_CLEAN'] = df['YEARS_EMPLOYED'].mask(df['DAYS_EMPLOYED'] == 365243, np.nan)

df['CREDIT_INCOME_RATIO']  = df['AMT_CREDIT']  / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

df[['AGE_YEARS', 'YEARS_EMPLOYED_CLEAN', 'CREDIT_INCOME_RATIO']].describe()

### The good/bad frame

Scorecard language calls the event you are trying to predict the **bad**, and everything else the **good**. In Home Credit, `TARGET == 1` is a payment-difficulty case, so:

- **bad**  = `TARGET == 1`  (the event / the default)
- **good** = `TARGET == 0`  (no event)

**Exercise 0.1** — Compute and print: the number of goods, number of bads, the overall **bad rate** (mean of `TARGET`), and the **odds** of good-to-bad in the full data. These four numbers are the reference points every WOE table is measured against.

**🧠 Conceptual 0.1** — The dataset is roughly 92% good / 8% bad. Nothing in the WOE or IV formulas divides by the number of rows in a bin as a fraction of *all* rows — they divide by the number of goods (out of all goods) and bads (out of all bads) separately. Why does that separate normalisation make WOE robust to the class imbalance, when a raw "bad rate per bin" is not?

### Train / validation split — do this before *any* WOE math

WOE and IV are **learned** quantities: each WOE value is a statistic estimated from data. If you estimate them on rows you later score, you have leaked the target into your features. So split first.

**Exercise 0.2** — Using `sklearn.model_selection.train_test_split`, make a **stratified** 80/20 split on `TARGET` into `train` and `valid` DataFrames (`random_state=42`). Print the shape and the bad rate of each split and confirm the bad rates match (that is what `stratify` buys you).

**🧠 Conceptual 0.2** — Suppose you (wrongly) computed WOE on the full data, transformed, then split for modelling. Your validation AUC would look *better* than the model deserves. Explain the mechanism — what specifically has leaked, and through which numbers?

In [ ]:
# Ex 0.1 — good/bad counts, bad rate, odds

In [ ]:
# Ex 0.2 — stratified train/valid split (all WOE math uses `train` only from here on)
from sklearn.model_selection import train_test_split

# train, valid = train_test_split(...)

**Your answers (0.1, 0.2):**

**0.1** —

**🧠 0.1** —

**🧠 0.2** —

## 1 — Weight of Evidence from first principles

WOE measures, for a single bin of a feature, **how the goods and bads are distributed relative to each other** — on the log scale.

For bin \(i\):

$$
\text{WOE}_i \;=\; \ln\!\left( \frac{\text{Distribution of Goods}_i}{\text{Distribution of Bads}_i} \right)
\;=\; \ln\!\left( \frac{g_i / G}{b_i / B} \right)
$$

where \(g_i, b_i\) are the goods/bads in bin \(i\), and \(G, B\) are the totals. Note the numerators are **within-class shares**, not counts: \(g_i/G\) is "what fraction of *all goods* fall in this bin".

**The convention (fix it now, obey it forever).** With **goods on top**:

- \(\text{WOE}_i > 0\) → the bin holds a *larger* share of goods than of bads → **safer than average**.
- \(\text{WOE}_i < 0\) → the bin is **riskier than average**.
- \(\text{WOE}_i = 0\) → the bin's good/bad mix equals the portfolio's.

The other convention puts bads on top and flips every sign. Neither is "right" — but this notebook uses **goods-on-top** everywhere. Higher WOE = safer.

**Key syntax — a WOE table by hand (your reference implementation)**

```python
# One categorical feature, computed on the TRAIN split only.
feat = "NAME_EDUCATION_TYPE"

ct = pd.crosstab(train[feat], train["TARGET"])      # rows = bins, cols = 0/1
ct.columns = ["good", "bad"]

G, B = ct["good"].sum(), ct["bad"].sum()

ct["dist_good"] = ct["good"] / G                    # within-class share of goods
ct["dist_bad"]  = ct["bad"]  / B                    # within-class share of bads
ct["woe"]       = np.log(ct["dist_good"] / ct["dist_bad"])
ct["bad_rate"]  = ct["bad"] / (ct["good"] + ct["bad"])
```

### Exercises

**1.1** — Reproduce the reference table above for `NAME_EDUCATION_TYPE` on `train`. Sort by `woe` and read it: which education level is safest, which riskiest, and does the ranking match your prior from 1.5?

**1.2** — Confirm the two sanity checks: `dist_good` sums to 1 and `dist_bad` sums to 1 (up to floating point). If they don't, you have a bug — what would cause a sum ≠ 1?

**1.3** — Add a `log_odds` column: \(\ln(g_i / b_i)\) (raw within-bin log odds of good-to-bad). Then compute the portfolio log-odds \(\ln(G/B)\). Show numerically that \(\text{WOE}_i = \text{log\_odds}_i - \ln(G/B)\). This identity is the whole point of WOE — write down what it means in words.

**Interpretation 1.1** — In your convention, a bin with WOE = +0.5 — is it safer or riskier than the portfolio, and by roughly what factor on the odds scale? (Hint: exponentiate.)

**Interpretation 1.2** — Why the logarithm at all? What breaks if you use the raw ratio \(\text{dist\_good}_i / \text{dist\_bad}_i\) as your evidence measure instead of its log? Think about symmetry: a bin twice as safe vs a bin twice as risky.

**🧠 Conceptual 1.3** — From Exercise 1.3 you showed WOE is just the bin's log-odds *minus a constant*. A logistic regression models \(\text{logit}(p) = \beta_0 + \sum \beta_j x_j\). Explain, in one or two sentences, why feeding **WOE-encoded** features into logistic regression is such a natural fit — what does each feature's contribution already look like before the model even fits a coefficient?

In [ ]:
# Ex 1.1 — WOE table for NAME_EDUCATION_TYPE (train only)

In [ ]:
# Ex 1.2 — sanity checks: dist_good and dist_bad each sum to 1

In [ ]:
# Ex 1.3 — WOE == within-bin log-odds minus portfolio log-odds

**Your answers (1.1–1.3):**

**I 1.1** —

**I 1.2** —

**🧠 1.3** —

## 2 — Information Value: how strong is the whole feature?

WOE describes one bin. **Information Value** collapses the whole feature into a single number — how much the bins, taken together, separate goods from bads:

$$
\text{IV} \;=\; \sum_i \left( \text{dist\_good}_i - \text{dist\_bad}_i \right) \times \text{WOE}_i
$$

Each bin contributes \((\text{dist\_good}_i - \text{dist\_bad}_i)\times \text{WOE}_i\), which is **always ≥ 0** (the sign of the gap and the sign of the WOE always agree). So IV is a sum of non-negative pieces — a feature can't have "cancelling" predictive power.

**Rule-of-thumb bands (Siddiqi):**

| IV | Interpretation |
|---|---|
| < 0.02 | Not predictive — drop |
| 0.02 – 0.1 | Weak |
| 0.1 – 0.3 | Medium |
| 0.3 – 0.5 | Strong |
| > 0.5 | **Suspiciously strong** — investigate for leakage / over-fit bins |

**Key syntax**

```python
ct["iv_contrib"] = (ct["dist_good"] - ct["dist_bad"]) * ct["woe"]
iv = ct["iv_contrib"].sum()
```

### Exercises

**2.1** — Add `iv_contrib` to your 1.1 education table and print the total IV. Which single bin contributes the most IV, and is it the highest-WOE bin or just a well-populated one?

**2.2** — Wrap it all in a reusable function:

```python
def woe_iv_table(data, feature, target="TARGET", smoothing=0.5):
    \"\"\"Return (table, iv) for a categorical/binned feature, goods-on-top convention.\"\"\"
    ...
```

It should return the per-bin table (counts, dists, woe, bad_rate, iv_contrib) and the scalar IV. You will reuse this constantly — get it right once.

**2.3** — The zero-cell problem: if a bin has 0 bads (or 0 goods), `dist_bad = 0` and WOE is \(\ln(\text{something}/0) = +\infty\). Add **smoothing**: replace \(g_i, b_i\) with \(g_i + 0.5\) and \(b_i + 0.5\) (or add `smoothing` to each) *before* forming the distributions. Rebuild the education table with and without smoothing and note where it matters.

**Interpretation 2.1** — IV throws away the *sign* of each WOE (the contribution is always positive). Give a concrete decision you can make from a WOE table that you cannot make from the IV number alone.

**Interpretation 2.2** — Why is IV > 0.5 a *red flag* rather than good news in a PD model? Name two distinct things that produce a suspiciously high IV, and how you'd tell them apart.

**🧠 Conceptual 2.3** — Two analysts bin the same feature. Analyst A uses 3 coarse bins and gets IV = 0.18. Analyst B uses 40 fine bins and gets IV = 0.34. B's IV is higher. Is B's feature "more predictive"? Explain what happens to IV as you add bins, and why maximising IV over binnings is the wrong objective.

In [ ]:
# Ex 2.1 — add IV contribution + total IV to the education table

In [ ]:
# Ex 2.2 — reusable woe_iv_table(data, feature, target='TARGET', smoothing=0.5)
def woe_iv_table(data, feature, target="TARGET", smoothing=0.5):
    ...

In [ ]:
# Ex 2.3 — smoothing / the zero-cell problem

**Your answers (2.1–2.3):**

**I 2.1** —

**I 2.2** —

**🧠 2.3** —

## 3 — Binning categorical variables (coarse classing)

Raw categoricals often have too many levels, some with a handful of rows. A WOE from 30 rows is noise. **Coarse classing** = merging levels into a smaller set of stable, sensible bins before computing WOE.

Rules of thumb: each final bin holds at least ~5% of the data (or a fixed floor like 200 rows), merge on **similar bad rate / WOE** *and* domain sense, and give **missing its own bin** — never silently drop it.

**Key syntax**

```python
# frequency of each level
train[feat].value_counts(dropna=False)

# collapse rare levels into "OTHER"
counts = train[feat].value_counts()
rare   = counts[counts < 0.05 * len(train)].index
train[feat + "_c"] = train[feat].where(~train[feat].isin(rare), other="OTHER")

# make NaN an explicit, WOE-able category
train[feat + "_c"] = train[feat + "_c"].astype("object").fillna("MISSING")

# merge specific levels by mapping
mapping = {"Businessman": "Working", "Student": "OTHER", ...}
train[feat + "_c"] = train[feat].replace(mapping)
```

### Exercises

**3.1** — `OCCUPATION_TYPE` has ~18 levels plus a big chunk of missing. On `train`: build the raw `woe_iv_table`, then coarse-class it — put `NaN` in its own `MISSING` bin, merge any level under 5% into sensible groups (use bad rate + your 1.5 occupation findings to decide what merges with what). Report IV before vs after coarse classing.

**3.2** — `NAME_INCOME_TYPE`: some levels (e.g. `Student`, `Businessman`, `Maternity leave`) have very few rows. Show their counts and bad rates, then justify a merge and rebuild the table.

**3.3** — Take one merge you made in 3.1 and *deliberately* do it badly (merge a high-risk level into a low-risk group). Recompute IV and the WOE of the merged bin. Quantify what the bad merge cost you.

**Interpretation 3.1** — What is the tension in choosing bin count? State what you lose by having bins that are *too fine* and what you lose when they are *too coarse*, in WOE/IV terms.

**Interpretation 3.2** — Missing `OCCUPATION_TYPE` is not missing-at-random (recall the 1.5 sentinel work — pensioners, etc.). Why is putting `MISSING` in its own WOE bin often *more* informative than imputing it? When could a `MISSING` bin itself leak?

**🧠 Conceptual 3.3** — You coarse-classed on `train`. A level `Maternity leave` appears in `valid` but you merged it into `OTHER` on train. When you transform `valid`, what WOE does a `Maternity leave` row receive, and what rule did you have to fix in advance to make that deterministic?

In [ ]:
# Ex 3.1 — OCCUPATION_TYPE: raw table, then coarse-classed (MISSING bin + rare merges)

In [ ]:
# Ex 3.2 — NAME_INCOME_TYPE: inspect rare levels, justify and apply a merge

In [ ]:
# Ex 3.3 — a deliberately bad merge, and what it costs in IV

**Your answers (3.1–3.3):**

**I 3.1** —

**I 3.2** —

**🧠 3.3** —

## 4 — Binning continuous variables (fine → coarse classing)

A continuous feature has no natural bins — you impose them. Two starting strategies:

- **Equal-frequency** (`pd.qcut`) — each bin has ~the same number of rows. Robust to outliers, gives every bin enough data for a stable WOE. Usually the right default for scorecards.
- **Equal-width** (`pd.cut`) — bins span equal ranges. Intuitive edges, but outliers leave some bins nearly empty.

Then **coarse-class**: merge adjacent fine bins so the WOE trend is smooth/monotonic. And, again: **missing gets its own bin.**

**Key syntax**

```python
train["EXT3_bin"] = pd.qcut(train["EXT_SOURCE_3"], q=10, duplicates="drop")   # deciles
train["AGE_bin"]  = pd.cut(train["AGE_YEARS"], bins=[20,25,30,40,50,60,100])  # explicit edges

# missing rows get their own labelled bin (qcut/cut leave NaN as NaN)
train["EXT3_bin"] = train["EXT3_bin"].cat.add_categories("MISSING").fillna("MISSING")

# then just feed the binned column to your woe_iv_table
tbl, iv = woe_iv_table(train, "EXT3_bin")
```

### Exercises

**4.1** — `EXT_SOURCE_3`: `qcut` into 10 bins on `train`, add a `MISSING` bin, run `woe_iv_table`. Is the WOE monotonic across the deciles? (It should be close — this is a strong feature.)

**4.2** — Same feature with `pd.cut` into 10 *equal-width* bins. Compare the row counts per bin and the IV with the `qcut` version. Which bins got starved, and why did IV change?

**4.3** — `AGE_YEARS`: bin with sensible business edges (e.g. 20-25-30-35-40-50-60-∞), compute the WOE table. Recall from 1.5 that young borrowers are riskier — does the WOE trend agree, and is it monotonic?

**4.4** — `CREDIT_INCOME_RATIO`: qcut into 8 bins (mind the outliers). Report IV and the WOE trend.

**Interpretation 4.1** — Equal-frequency vs equal-width: for a heavily right-skewed money variable, which gives more trustworthy WOEs and why? What does the starved-bin problem do to a WOE estimate?

**Interpretation 4.2** — Missing `EXT_SOURCE_3` — why bin it separately rather than dropping the rows or imputing the median *before* binning? What would median-imputation do to that bin's WOE and to the feature's IV?

**🧠 Conceptual 4.3** — You qcut `EXT_SOURCE_3` into deciles on `train`, getting bin edges from the training quantiles. To transform `valid`, do you (a) recompute quantile edges on `valid`, or (b) reuse the train edges? One of these is a leak / an inconsistency. Say which and why, and what happens to a `valid` value that falls outside the train min/max.

In [ ]:
# Ex 4.1 — EXT_SOURCE_3 qcut into 10 + MISSING bin, WOE table

In [ ]:
# Ex 4.2 — EXT_SOURCE_3 equal-width cut, compare counts + IV with qcut

In [ ]:
# Ex 4.3 — AGE_YEARS with business edges, WOE trend + monotonicity

In [ ]:
# Ex 4.4 — CREDIT_INCOME_RATIO qcut into 8, IV + trend

**Your answers (4.1–4.4):**

**I 4.1** —

**I 4.2** —

**🧠 4.3** —

## 5 — Monotonic binning

For a scorecard (logistic regression on WOE), you usually *want* the WOE to move **monotonically** across an ordered feature's bins: as the score rises, risk should fall (or rise) without zig-zags. Monotonic WOE gives a scorecard that is interpretable ("more X always means safer") and resists over-fitting the wiggles of a particular sample.

The workflow: **fine-class** (many bins) → check the trend → **merge adjacent bins** until the WOE is monotonic (or the trend is smooth enough), subject to a minimum bin size.

**Key syntax**

```python
# quick monotonic-trend check: rank-correlation of bin order vs bin WOE
from scipy.stats import spearmanr
rho, _ = spearmanr(np.arange(len(tbl)), tbl["woe"].values)

# merge two adjacent bins by re-labelling, then recompute the table
# (many teams instead reach for a library — see Section 9 — but do it by hand once)
```

### Exercises

**5.1** — Take your 4.1 `EXT_SOURCE_3` decile table. Compute the Spearman correlation between bin index and WOE as a monotonicity score. If any decile breaks the trend, merge it with a neighbour and recompute until monotonic. Report bins before/after and the IV cost (if any) of enforcing monotonicity.

**5.2** — `AGE_YEARS` risk is often **U-shaped-ish** or monotone-ish but not perfectly. Fine-bin it (~10 bins), inspect the WOE trend, and decide: can you make it monotonic with sensible merges, or is a non-monotonic pattern real here? Defend your choice.

**5.3** — Write a small helper `is_monotonic(woe_series)` that returns whether a WOE vector is entirely non-increasing or non-decreasing, and use it to flag which of your Section 4 features are already monotonic.

**Interpretation 5.1** — Why does a logistic-regression scorecard prefer monotonic WOE? What interpretability and stability problems does a zig-zag WOE bin create for the people who *use* the scorecard?

**Interpretation 5.2** — Enforcing monotonicity almost always *lowers* IV a little. Why is accepting that small IV loss usually the right call? When would you refuse to force monotonicity?

**🧠 Conceptual 5.3** — Over-binning can produce a beautiful, perfectly monotonic WOE curve on `train` with a high IV — that then falls apart on `valid`. Explain how monotonicity and high in-sample IV can *coexist with* overfitting, and what out-of-sample check catches it (you'll build it in Section 8).

In [ ]:
# Ex 5.1 — enforce monotonic WOE on EXT_SOURCE_3 by merging bins
from scipy.stats import spearmanr

In [ ]:
# Ex 5.2 — AGE_YEARS: is the non-monotonic pattern real? decide and defend

In [ ]:
# Ex 5.3 — is_monotonic(woe_series) helper + apply to Section 4 features
def is_monotonic(woe):
    ...

**Your answers (5.1–5.3):**

**I 5.1** —

**I 5.2** —

**🧠 5.3** —

## 6 — WOE encoding: transforming features for the model

Once a feature is binned and its WOE table is settled **on train**, you *encode* it: replace each row's bin with that bin's WOE number. The model then sees a clean, monotone, log-odds-aligned numeric column.

The discipline that matters: **the mapping is fitted on `train` and merely applied to `valid`.** Unseen categories and out-of-range values in `valid` must map to something you decided in advance (commonly WOE = 0, i.e. "neutral / no evidence", or the WOE of a designated `OTHER`/`MISSING` bin).

**Key syntax**

```python
tbl, iv = woe_iv_table(train, "EXT3_bin")
woe_map = tbl["woe"].to_dict()               # {bin_label: woe}

def apply_woe(series_binned, woe_map, default=0.0):
    return series_binned.map(woe_map).fillna(default)

train["EXT3_woe"] = apply_woe(train["EXT3_bin"], woe_map)
valid["EXT3_bin"] = pd.cut(valid["EXT_SOURCE_3"], bins=train_edges)   # SAME edges
valid["EXT3_woe"] = apply_woe(valid["EXT3_bin"], woe_map, default=0.0)
```

### Exercises

**6.1** — Encode `EXT_SOURCE_3` (from your monotonic bins in Section 5) into `EXT3_woe` on **both** `train` and `valid`, reusing the train edges and the train WOE map. Confirm `valid` never influenced a WOE number.

**6.2** — Encode your coarse-classed `OCCUPATION_TYPE` and `NAME_EDUCATION_TYPE`. Handle a category present in `valid` but absent in `train` by routing it to your `default`. Show it works by checking for NaNs after mapping (there should be none).

**6.3** — Fit a `LogisticRegression` on a handful of WOE-encoded features (`EXT_SOURCE_3`, `EXT_SOURCE_2`, age, occupation, education) using `train`, and report train vs valid ROC AUC. Then print the fitted coefficients.

**Interpretation 6.1** — After WOE encoding, the logistic coefficient on a well-behaved feature should be **positive and not far from a shared value** across features. Why? (Recall Section 1: WOE is already on the log-odds scale.) What does a *negative* coefficient on a WOE feature warn you about?

**Interpretation 6.2** — Mapping unseen `valid` categories to WOE = 0 means "this bin gives no evidence either way". What is the risk of that default if the unseen category is actually high-risk, and how would monitoring catch it?

**🧠 Conceptual 6.3** — WOE encoding replaces a category with a single number derived from the target. That is a form of **target encoding**. Explain precisely why doing it on `train` only (and applying to `valid`) avoids the target-leakage that naive target encoding is infamous for — and what within-`train` trick (e.g. cross-fitting / out-of-fold) you'd add if `train` itself were small.

In [ ]:
# Ex 6.1 — encode EXT_SOURCE_3 to WOE on train AND valid (train-learned map/edges)

In [ ]:
# Ex 6.2 — encode occupation + education; route unseen valid categories to default

In [ ]:
# Ex 6.3 — logistic regression on WOE features; train vs valid AUC + coefficients
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

**Your answers (6.1–6.3):**

**I 6.1** —

**I 6.2** —

**🧠 6.3** —

## 7 — Visualising WOE & IV

A WOE table is easier to *trust* as a picture. Three plots earn their keep: WOE-by-bin (is it monotonic?), event-rate-by-bin overlaid (does WOE track risk as expected?), and an IV ranking bar chart for feature selection.

> Read the **dataviz** skill before styling these — one consistent look across all your WOE plots.

**Key syntax**

```python
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(tbl)), tbl["woe"], color="tab:blue", alpha=0.8)
ax.axhline(0, color="black", linewidth=1)            # WOE=0 = portfolio average
ax.set_xticks(range(len(tbl)), [str(i) for i in tbl.index], rotation=45, ha="right")

ax2 = ax.twinx()                                      # overlay bad rate
ax2.plot(range(len(tbl)), tbl["bad_rate"], color="tab:red", marker="o")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
```

### Exercises

**7.1** — WOE bar chart for `EXT_SOURCE_3` bins with a horizontal line at WOE = 0. Order bins by their natural interval order. Title it with the takeaway.

**7.2** — Same chart, but overlay the **bad rate** per bin on a twin axis. Confirm visually that (in goods-on-top convention) WOE and bad rate move in *opposite* directions. Explain in the title or a note why that is expected.

**7.3** — Compute IV for a candidate list of ~10 features (mix of your binned continuous and coarse-classed categoricals) and draw a **horizontal, sorted IV ranking bar chart**, with the Siddiqi band cutoffs (0.02, 0.1, 0.3, 0.5) drawn as reference lines.

**Interpretation 7.1** — What does a *zig-zag* WOE bar chart tell you to do before this feature goes near the model? What does a smooth monotonic one let you claim to a credit committee?

**🧠 Conceptual 7.2** — In your 7.2 twin-axis chart, WOE goes up while bad rate goes down. If someone hands you a WOE chart where WOE and bad rate move in the *same* direction, what has almost certainly happened — and is the feature wrong, or just the label?

In [ ]:
# Ex 7.1 — WOE bar chart for EXT_SOURCE_3 bins, line at WOE=0

In [ ]:
# Ex 7.2 — WOE bars + bad-rate line on twin axis

In [ ]:
# Ex 7.3 — sorted IV ranking bar chart with Siddiqi band cutoffs

**Your answers (7.1, 7.2):**

**I 7.1** —

**🧠 7.2** —

## 8 — Pitfalls: leakage, stability (PSI), and over-binning

The formulas are easy; the judgement is the job. Three failure modes to build defences against.

**Stability — Population Stability Index.** PSI uses the *same math shape* as IV, but compares **the same feature across two populations** (train vs valid, or last-quarter vs this-quarter) instead of goods vs bads:

$$
\text{PSI} \;=\; \sum_i \left( \%\text{Actual}_i - \%\text{Expected}_i \right)\,\ln\!\left(\frac{\%\text{Actual}_i}{\%\text{Expected}_i}\right)
$$

Bands: **< 0.1** stable, **0.1–0.25** some shift (watch), **> 0.25** significant shift (act).

**Key syntax**

```python
def psi(expected_binned, actual_binned):
    e = expected_binned.value_counts(normalize=True)
    a = actual_binned.value_counts(normalize=True).reindex(e.index).fillna(1e-6)
    return float(((a - e) * np.log(a / e)).sum())
```

### Exercises

**8.1** — Compute PSI for `EXT_SOURCE_3` bins between `train` (expected) and `valid` (actual), reusing your train edges. Interpret against the bands. (A clean random split should be very stable — near 0.)

**8.2** — Over-binning demo: qcut `EXT_SOURCE_2` into **50** bins on `train`, compute IV on `train`, then apply the *same* bins+WOE map to `valid` and recompute a validation IV. Compare. Then do the same with **8** bins. What does the 50-bin gap tell you?

**8.3** — Leakage hunt: scan a handful of features for a **suspiciously high IV** (> 0.5). Pick one flagged feature (if any) and reason about whether it is genuine signal or leakage. If none in your list exceed 0.5, say what a leaky feature would look like here (e.g. a field only populated post-decision).

**Interpretation 8.1** — IV and PSI are the same arithmetic. State crisply what each *holds fixed* and what each *compares*, and why a feature can have high IV **and** high PSI (predictive but unstable) — and why you might still reject it.

**Interpretation 8.2** — From 8.2: why does train IV rise as you add bins while valid IV eventually falls? Name the bias–variance story in WOE terms.

**🧠 Conceptual 8.3** — "Reject inference": your training data only contains *accepted* applicants (the rejected ones never got a loan, so have no `TARGET`). How does this bias the WOE/IV you estimate, and in which direction would it distort the WOE of a risk-segregating feature? (Conceptual — no code required.)

In [ ]:
# Ex 8.1 — PSI of EXT_SOURCE_3 bins, train (expected) vs valid (actual)
def psi(expected_binned, actual_binned):
    ...

In [ ]:
# Ex 8.2 — over-binning: 50 bins vs 8 bins, train IV vs valid IV

In [ ]:
# Ex 8.3 — scan for IV > 0.5; reason about leakage vs genuine signal

**Your answers (8.1–8.3):**

**I 8.1** —

**I 8.2** —

**🧠 8.3** —

## 9 — Doing it with a library (`optbinning`)

You have now done WOE/IV by hand — which is the only way to actually understand it. In production, teams use a solver that chooses bin edges to **maximise IV subject to constraints** (minimum bin size, monotonic trend, max bins). `optbinning` is the common choice (`scorecardpy` is another).

```python
# pip install optbinning   (run once in your env if missing)
from optbinning import OptimalBinning

optb = OptimalBinning(name="EXT_SOURCE_3", dtype="numerical",
                      monotonic_trend="auto", min_bin_size=0.05)
optb.fit(train["EXT_SOURCE_3"].values, train["TARGET"].values)

optb.binning_table.build()          # WOE/IV table
train_woe = optb.transform(train["EXT_SOURCE_3"].values, metric="woe")
valid_woe = optb.transform(valid["EXT_SOURCE_3"].values, metric="woe")   # train-learned bins
```

> If `optbinning` will not install in your environment, skip the code but still answer the interpretation questions — and note what you would have compared.

### Exercises

**9.1** — `OptimalBinning` on `EXT_SOURCE_3` with `min_bin_size=0.05` and a monotonic trend. Compare its bins, WOE values, and IV against your **hand-built monotonic** version from Section 5. Where do they agree/differ?

**9.2** — Use `BinningProcess` to bin a *list* of ~8 features at once, pull out the IV of each, and rank them. Compare the ranking to your hand-computed 7.3 ranking.

**9.3** — Re-fit 9.1 with `monotonic_trend=None` and a larger `max_n_bins`. Does IV go up? Does the extra IV survive on `valid`? Tie this back to Section 8.

**Interpretation 9.1** — The solver "maximises IV subject to constraints". Why are the *constraints* (min bin size, monotonic trend, max bins) the important part — what would unconstrained IV-maximisation produce?

**🧠 Conceptual 9.2** — A colleague reports IV = 0.61 for a feature straight from `optbinning` with default settings and wants to include it. Walk through the three checks you would run before believing it is a great feature rather than a red flag.

In [ ]:
# Ex 9.1 — OptimalBinning on EXT_SOURCE_3 vs your hand-built monotonic bins
# from optbinning import OptimalBinning

In [ ]:
# Ex 9.2 — BinningProcess over ~8 features; IV ranking vs your 7.3 ranking
# from optbinning import BinningProcess

In [ ]:
# Ex 9.3 — unconstrained (monotonic_trend=None, more bins): train IV vs valid IV

**Your answers (9.1, 9.2):**

**I 9.1** —

**🧠 9.2** —

## 10 — Capstone: a WOE/IV feature-selection table for the PD model

Everything above, combined into the artefact you actually hand off: a ranked, cleaned, WOE-encoded feature set ready for a scorecard. No new syntax — this is composition, discipline, and judgement.

**The brief**

Working **on `train` only** for all learning, and applying to `valid`:

1. **Candidate set** — assemble ~12 features (a mix): `EXT_SOURCE_1/2/3`, `AGE_YEARS`, `YEARS_EMPLOYED_CLEAN`, `CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `NAME_EDUCATION_TYPE`, `OCCUPATION_TYPE`, `NAME_INCOME_TYPE`, `NAME_FAMILY_STATUS`, `REGION_RATING_CLIENT`.
2. **Bin each** — qcut/monotonic for continuous, coarse-class for categorical, `MISSING` bins throughout.
3. **Build a summary table** — one row per feature: `n_bins`, `IV`, IV band (Siddiqi), monotonic? (Y/N), PSI(train→valid). Sort by IV descending.
4. **Select** — apply a written rule (e.g. keep 0.02 ≤ IV ≤ 0.5, PSI < 0.25, drop one of any pair that is redundant — check with a quick correlation of the WOE-encoded columns).
5. **Encode** the selected features to WOE on `train` and `valid`, and fit a `LogisticRegression`; report train/valid AUC.
6. **One figure** — a sorted IV ranking bar chart of the candidates with the band cutoffs marked (reuse 7.3), saved to PNG at 150 dpi.

**Then write the analysis** (markdown cell below): a **decision statement per selected/rejected feature** in *observation → decision* form.

> Example: "EXT_SOURCE_3: IV 0.34 (strong), monotonic, PSI 0.01 → keep, WOE-encoded, expect a large scorecard weight. EXT_SOURCE_1: IV 0.15 but 56% missing and PSI 0.08 → keep with an explicit MISSING bin, flag the missingness to the committee."

**Requirements:** all WOE/IV/PSI learned on `train`; every feature has a `MISSING` bin where relevant; the summary table and the figure are both produced; the selection rule is written down *before* you read the table.

In [ ]:
# Capstone — build the WOE/IV feature-selection table here.
# (Develop each feature's binning in its own scratch cell first, then assemble the summary.)

**Capstone decision statements:**

Selection rule (written before reading the table):

Feature-by-feature (observation → decision):

- EXT_SOURCE_3:
- EXT_SOURCE_2:
- EXT_SOURCE_1:
- AGE_YEARS:
- YEARS_EMPLOYED_CLEAN:
- CREDIT_INCOME_RATIO:
- ANNUITY_INCOME_RATIO:
- NAME_EDUCATION_TYPE:
- OCCUPATION_TYPE:
- NAME_INCOME_TYPE:
- NAME_FAMILY_STATUS:
- REGION_RATING_CLIENT:

Model result (train vs valid AUC) and what it says about the selected set:

## 11 — Self-test: can you answer these cold?

Close the notebook docs. If any of these are shaky, redo the relevant section.

1. Write the WOE formula and state your sign convention. In it, does a **high WOE** mean safer or riskier?
2. Why are the distributions computed *within class* (goods out of all goods, bads out of all bads) rather than as a share of all rows? What does that buy you under class imbalance?
3. Write the IV formula. Why is every bin's contribution non-negative, and why is that both convenient and a limitation?
4. State the Siddiqi IV bands. Why is IV > 0.5 a warning rather than a prize?
5. WOE = (bin log-odds) − (portfolio log-odds). Why does that identity make WOE-encoded features ideal for logistic regression?
6. Equal-frequency vs equal-width binning — when do you reach for each, and what is the "starved bin" failure?
7. Why give missing values their own WOE bin instead of imputing before binning?
8. Why do scorecards prefer monotonic WOE, and what do you sacrifice to enforce it?
9. Everything WOE/IV must be learned on `train` only. Describe the exact leak that happens if you compute WOE on all data before splitting.
10. IV and PSI are the same arithmetic — what does each hold fixed and compare, and what are the PSI action bands?
11. WOE encoding is target encoding. Why does the train-only discipline defuse the usual target-leakage objection?
12. Give one case where a *non-monotonic* WOE pattern is genuinely real and you should keep it.

**When you're done:** bring your completed sections to Claude for marking — the WOE/IV tables, the plots, and the written answers. This feeds straight into building the logistic scorecard for the PD model.

### Appendix — a compact reference implementation

Once you have built these by hand, this is a tidy version worth keeping. Fill in / verify it against your own Section 2 function rather than copying blindly.

In [ ]:
# Reference WOE/IV utility (goods-on-top convention). Verify against your own before trusting it.
def woe_iv(data, feature, target="TARGET", smoothing=0.5):
    g = data.groupby(feature, dropna=False)[target]
    t = pd.DataFrame({"good": g.apply(lambda s: (s == 0).sum()),
                      "bad":  g.apply(lambda s: (s == 1).sum())})
    G = t["good"].sum() + smoothing * len(t)
    B = t["bad"].sum()  + smoothing * len(t)
    t["dist_good"] = (t["good"] + smoothing) / G
    t["dist_bad"]  = (t["bad"]  + smoothing) / B
    t["woe"]       = np.log(t["dist_good"] / t["dist_bad"])
    t["bad_rate"]  = t["bad"] / (t["good"] + t["bad"])
    t["iv"]        = (t["dist_good"] - t["dist_bad"]) * t["woe"]
    return t, t["iv"].sum()

# tbl, iv = woe_iv(train, "NAME_EDUCATION_TYPE")
# display(tbl.sort_values("woe")); print("IV:", round(iv, 4))